# Azure AI & RAG Practice Notebook

This notebook contains skeleton code to practice building RAG pipelines using Azure OpenAI and LangChain.

## 1. Setup Environment
You will need an Azure OpenAI Endpoint and API Key.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

load_dotenv()

# Set environment variables
os.environ["AZURE_OPENAI_API_KEY"] = "your-key"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource.openai.azure.com/"
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-02-01"
os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"] = "gpt-4o"
os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME"] = "text-embedding-3-small"

## 2. Document Ingestion
Practice loading a PDF and chunking it.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def ingest_docs(file_path):
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    splits = text_splitter.split_documents(docs)
    
    print(f"Created {len(splits)} chunks from {file_path}")
    return splits

# splits = ingest_docs("path/to/your/document.pdf")

## 3. Vector Store (Azure AI Search Mockup)
For local practice, we can use FAISS, but remember the JD asks for Azure AI Search.

In [ ]:
from langchain_community.vectorstores import FAISS

def create_vector_store(splits):
    embeddings = AzureOpenAIEmbeddings(
        azure_deployment=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME"],
        openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"]
    )
    vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
    return vectorstore

# vectorstore = create_vector_store(splits)

## 4. Retrieval & RAG Chain
Building the chain to answer questions.

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

def build_rag_chain(vectorstore):
    llm = AzureChatOpenAI(
        azure_deployment=os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"],
        openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
        temperature=0
    )
    
    system_prompt = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer the question. "
        "If you don't know the answer, just say that you don't know. "
        "Use three sentences maximum and keep the answer concise."
        "\n\n"
        "{context}"
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])
    
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(vectorstore.as_retriever(), question_answer_chain)
    
    return rag_chain

# chain = build_rag_chain(vectorstore)
# response = chain.invoke({"input": "What is the main topic of the document?"})
# print(response["answer"])